In [1]:
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score

df = pd.read_csv("../data/kbbi_v.csv")
print(f"Entri berlabel kelas: {df['kelas'].notna().sum():,} dari {len(df):,}")

Entri berlabel kelas: 116,357 dari 146,280


In [2]:
# fitur (X) dan label (y), sampel hasil Konsep Dasar Machine Learning
data = pd.read_csv("../data/kbbi_sampel.csv")

X = data["submakna"]
y = data["kelas_utama"]
pd.DataFrame({"X (makna)": X.head(4).str[:60].values, "y (kelas)": y.head(4).values})

,X (makna),y (kelas)
0,lalai; alpa,Adjektiva
1,"cenderung menyumbat pori-pori, terutama pada s...",Adjektiva
2,antara desa yang satu dengan desa yang lainnya...,Adjektiva
3,deras (tentang arus air),Adjektiva


In [3]:
# klasifikasi: 6 langkah supervised learning
# 1. data sudah siap (X dan y)
# 2. bagi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
vec = TfidfVectorizer(max_features=3000, min_df=2)
X_train_v = vec.fit_transform(X_train)
X_test_v = vec.transform(X_test)

# 3. pilih model
model = LogisticRegression(max_iter=1000)

# 4. latih
model.fit(X_train_v, y_train)

# 5. evaluasi
print(f"Akurasi: {accuracy_score(y_test, model.predict(X_test_v)):.1%}")

# 6. gunakan pada data baru
baru = ["orang yang mengajar di sekolah", "melakukan sesuatu dengan cepat", "tidak memiliki sifat yang baik"]
for makna, kelas in zip(baru, model.predict(vec.transform(baru))):
    print(f"{makna} => {kelas}")

Akurasi: 71.4%
orang yang mengajar di sekolah => Nomina
melakukan sesuatu dengan cepat => Verba
tidak memiliki sifat yang baik => Adjektiva


In [4]:
# simpan model dan pembagian data untuk dipakai di Unsupervised Learning
data["split"] = "train"
data.loc[X_test.index, "split"] = "test"
data.to_csv("../data/kbbi_sampel.csv", index=False)
joblib.dump((vec, model), "../models/model_kelas.joblib")

['../models/model_kelas.joblib']

In [5]:
# regresi: tebak jumlah huruf dari jumlah suku kata (kolom nama, contoh me.rem.bes)
kata = df["nama"].dropna()
kata = kata[kata.str.match(r"^[a-z]+(\.[a-z]+)+$")]

reg = pd.DataFrame({"suku_kata": kata.str.count(r"\.") + 1,
                    "huruf": kata.str.replace(".", "", regex=False).str.len()})

Xr_train, Xr_test, yr_train, yr_test = train_test_split(reg[["suku_kata"]], reg["huruf"], test_size=0.2, random_state=42)
reg_model = LinearRegression().fit(Xr_train, yr_train)
pred = reg_model.predict(Xr_test)

print(f"MAE: {mean_absolute_error(yr_test, pred):.2f} huruf")
print(f"R2: {r2_score(yr_test, pred):.3f}")
print(f"Kata 4 suku kata diperkirakan punya {reg_model.predict(pd.DataFrame({'suku_kata': [4]}))[0]:.1f} huruf")

MAE: 0.99 huruf
R2: 0.762
Kata 4 suku kata diperkirakan punya 9.5 huruf


In [6]:
# pentingnya data uji: model yang menghafal
hafal = DecisionTreeClassifier(random_state=42).fit(X_train_v, y_train)
print(f"Akurasi data latih: {accuracy_score(y_train, hafal.predict(X_train_v)):.1%}")
print(f"Akurasi data uji: {accuracy_score(y_test, hafal.predict(X_test_v)):.1%}")

Akurasi data latih: 95.6%
Akurasi data uji: 65.2%


In [7]:
# refleksi: klasifikasi atau regresi?
pd.DataFrame({
    "Skenario": [1, 2, 3, 4, 5],
    "Supervised?": ["Ya", "Ya", "Ya", "Tidak (Unsupervised, Clustering)", "Ya"],
    "Jenis": ["Klasifikasi", "Regresi", "Regresi", "Clustering", "Klasifikasi"],
    "Label": ["Gagal bayar (Ya/Tidak)", "Harga apartemen (Rp)", "Lama rawat ICU (hari)", "Tidak ada", "Kategori produk"],
})

,Skenario,Supervised?,Jenis,Label
0,1,Ya,Klasifikasi,Gagal bayar (Ya/Tidak)
1,2,Ya,Regresi,Harga apartemen (Rp)
2,3,Ya,Regresi,Lama rawat ICU (hari)
3,4,"Tidak (Unsupervised, Clustering)",Clustering,Tidak ada
4,5,Ya,Klasifikasi,Kategori produk


## Analisis

Dari 146.280 entri KBBI, hanya 116.357 (sekitar 79,5 persen) yang punya label kelas, jadi hanya data inilah yang bisa dipakai untuk supervised learning. Model klasifikasi mencapai akurasi 71,4 persen dan berhasil menebak tiga kalimat makna baru dengan benar: orang yang mengajar di sekolah sebagai nomina, melakukan sesuatu dengan cepat sebagai verba, dan tidak memiliki sifat yang baik sebagai adjektiva. Pada regresi, jumlah suku kata dipakai untuk menebak jumlah huruf dengan kesalahan rata-rata (MAE) 0,99 huruf dan R2 0,762, artinya sekitar 76 persen variasi jumlah huruf bisa dijelaskan oleh jumlah suku kata. Kata dengan 4 suku kata diperkirakan punya sekitar 9,5 huruf.

Bagian yang paling penting adalah percobaan model yang menghafal. Decision tree mencapai akurasi 95,6 persen pada data latih tetapi hanya 65,2 persen pada data uji, selisih 30,4 poin, dan hasil ujinya lebih rendah daripada regresi logistik yang lebih sederhana (71,4 persen). Ini membuktikan bahwa skor pada data latih tidak bisa dipercaya, dan data uji yang tidak pernah dilihat model adalah satu satunya ukuran yang jujur.